# **Assignment 4: some database operations**
- **you will learn:** using joins, aggregation, groupby in Pandas
- **task:**  See section 4.2 below
- **deadline:** 15.11.2026
- [Pandas documentation](https://pandas.pydata.org/docs/index.html)
- 📝 **Reminder:** Sync your GitHub repository with the main course repository, update your project in PyCharm, and after completing the assignment, commit and push your changes back to GitHub.

In [1]:
import numpy as np
import pandas as pd

print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

NumPy version: 2.4.6
Pandas version: 3.0.6


In [2]:
# -------------------------------
# 1) Build synthetic datasets
# -------------------------------
# sensor readouts (long-format)
np.random.seed(42)
N_SENSORS = 6
sensors = [f'S{i+1}' for i in range(N_SENSORS)]

start = pd.Timestamp('2025-11-01 00:00')
periods = 24 * 6 # six measurements per hour for 24 hours
freq = '10min' # 10-minute frequency
idx = pd.date_range(start=start, periods=periods, freq=freq)

rows = []
for sensor in sensors:
    base_temp = 15 + 5 * np.random.rand() # baseline temp different per sensor
    base_hum = 40 + 20 * np.random.rand()
    noise_t = np.random.normal(0, 1.5, size=len(idx))
    noise_h = np.random.normal(0, 5.0, size=len(idx))
    # add a diurnal temperature variation
    hours = (idx.hour + idx.minute/60.0)
    diurnal = 6 * np.sin(2 * np.pi * (hours - 6) / 24)
    
    temps = base_temp + diurnal + noise_t
    hums = np.clip(base_hum - 0.5 * diurnal + noise_h, 0, 100)
    
    for ts, t, h in zip(idx, temps, hums):
        rows.append({'sensor_id': sensor, 'timestamp': ts, 'temperature': t, 'humidity': h})

sensor_df = pd.DataFrame(rows)

# add some missing values intentionally and some outliers
sensor_df.loc[sensor_df.sample(frac=0.01, random_state=1).index, 'temperature'] = np.nan
sensor_df.loc[sensor_df.sample(frac=0.005, random_state=2).index, 'humidity'] = np.nan
# inject outliers
outlier_idx = sensor_df.sample(frac=0.002, random_state=3).index
sensor_df.loc[outlier_idx, 'temperature'] += np.random.choice([20, -15], size=len(outlier_idx))

print('--- sensor_df (head) ---')
print(sensor_df.head())

# sensor metadata (wide)
meta = pd.DataFrame({
'sensor_id': sensors,
'location': ['Room A', 'Room B', 'Room C', 'Room A', 'Room D', 'Room B'],
'type': ['thermo-hygro', 'thermo-hygro', 'thermo', 'thermo-hygro', 'hygro', 'thermo']
})

# events table (sparse)
events = pd.DataFrame([
{'sensor_id': 'S1', 'timestamp': start + pd.Timedelta(hours=5), 'event': 'maintenance'},
{'sensor_id': 'S3', 'timestamp': start + pd.Timedelta(hours=8, minutes=20), 'event': 'calibration'},
{'sensor_id': 'S2', 'timestamp': start + pd.Timedelta(hours=18), 'event': 'spike_detected'}
])

--- sensor_df (head) ---
  sensor_id           timestamp  temperature   humidity
0        S1 2025-11-01 00:00:00    11.844233  55.829533
1        S1 2025-11-01 00:10:00    13.162956  55.409148
2        S1 2025-11-01 00:20:00    10.544302  64.612578
3        S1 2025-11-01 00:30:00          NaN  63.473544
4        S1 2025-11-01 00:40:00    13.332673  63.221174


In [3]:
# -------------------------------
# 2) MultiIndex: set index as (sensor_id, timestamp)
# -------------------------------
sensor_mi = sensor_df.set_index(['sensor_id', 'timestamp']).sort_index()
print('\n--- sensor_mi (index sample) ---')
print(sensor_mi.head())

# show selecting using MultiIndex
print('\nSelect all readings for S2 between two times:')
print(sensor_mi.loc['S2'].between_time('06:00', '09:00').head())


--- sensor_mi (index sample) ---
                               temperature   humidity
sensor_id timestamp                                  
S1        2025-11-01 00:00:00    11.844233  55.829533
          2025-11-01 00:10:00    13.162956  55.409148
          2025-11-01 00:20:00    10.544302  64.612578
          2025-11-01 00:30:00          NaN  63.473544
          2025-11-01 00:40:00    13.332673  63.221174

Select all readings for S2 between two times:
                     temperature   humidity
timestamp                                  
2025-11-01 06:00:00    16.292123  38.803557
2025-11-01 06:10:00    16.560178  53.083026
2025-11-01 06:20:00    15.860508  54.342884
2025-11-01 06:30:00    14.793219  44.742754
2025-11-01 06:40:00    16.511539  48.741353


In [4]:
# -------------------------------
# 3) Multi-level columns example: create aggregated summary with MultiIndex columns
# -------------------------------
agg_funcs = {
'temperature': ['mean', 'std', 'min', 'max'],
'humidity': ['mean', 'std']
}
summary_by_sensor = sensor_df.groupby('sensor_id').agg(agg_funcs)
# flatten/unflatten demonstration: keep as MultiIndex columns
print('\n--- summary_by_sensor (MultiIndex columns) ---')
print(summary_by_sensor.head())

# rename columns to readable MultiIndex (level names)
summary_by_sensor.columns.names = ['measurement', 'stat']
print('\nColumns levels:', summary_by_sensor.columns.names)

# access a specific subtable using xs
print('\nTemperature means:')
print(summary_by_sensor.xs('temperature', axis=1)[['mean', 'std']])


--- summary_by_sensor (MultiIndex columns) ---
          temperature                                  humidity          
                 mean       std       min        max       mean       std
sensor_id                                                                
S1          16.718369  4.511676  8.552629  25.213955  59.259889  5.560005
S2          15.427998  4.497470  2.122613  24.023564  45.599224  5.493384
S3          15.658394  4.421943  7.273332  25.556400  56.385015  5.146519
S4          15.668049  4.238744  7.365955  24.204889  48.812680  5.493305
S5          17.801163  4.889507  9.133021  38.594542  49.913718  5.367881

Columns levels: ['measurement', 'stat']

Temperature means:
stat            mean       std
sensor_id                     
S1         16.718369  4.511676
S2         15.427998  4.497470
S3         15.658394  4.421943
S4         15.668049  4.238744
S5         17.801163  4.889507
S6         18.627710  4.633420


In [ ]:
summary_by_sensor.columns = [
    f"{measurement}_{stat}"
    for measurement, stat in summary_by_sensor.columns
]

print('\n--- summary_by_sensor (flat columns) ---')
print(summary_by_sensor.head())

In [5]:
# -------------------------------
# 4) Advanced groupby with named aggregation and custom functions
# -------------------------------
# We'll compute per-sensor and per-location aggregations using named aggregations
sensor_with_meta = sensor_df.merge(meta, on='sensor_id', how='left')

agg_named = sensor_with_meta.groupby(['location']).agg(
temp_mean=('temperature', 'mean'),
temp_med=('temperature', 'median'),
temp_iqr=('temperature', lambda x: np.subtract(*np.percentile(x.dropna(), [75, 25]))),
humidity_mean=('humidity', 'mean'),
n_readings=('temperature', 'count')
)
print('\n--- agg_named by location ---')
print(agg_named)

# custom aggregation using apply (slower but expressive) -> compute time-of-day sensitivity

def day_night_diff(group):
    group = group.copy()
    # daytime mean vs nighttime mean
    group['hour'] = group['timestamp'].dt.hour
    daytime = group.loc[(group['hour'] >= 6) & (group['hour'] < 18), 'temperature']
    nighttime = group.loc[(group['hour'] < 6) | (group['hour'] >= 18), 'temperature']
    return pd.Series({'day_minus_night': daytime.mean() - nighttime.mean()})

sens_daynight = sensor_df.merge(meta, on='sensor_id').groupby('sensor_id').apply(day_night_diff, include_groups=False)
print('\n--- day_minus_night per sensor ---')
print(sens_daynight.head())


--- agg_named by location ---
          temp_mean   temp_med  temp_iqr  humidity_mean  n_readings
location                                                           
Room A    16.187642  16.260233  7.800304      54.036285         283
Room B    17.022280  16.570175  7.810141      48.698361         287
Room C    15.658394  15.917286  8.167460      56.385015         143
Room D    17.801163  18.022922  8.284296      49.913718         142

--- day_minus_night per sensor ---
           day_minus_night
sensor_id                 
S1                7.691988
S2                7.222448
S3                7.461552
S4                7.165945
S5                7.980190


In [ ]:
# Note on merge vs join
# pandas.merge() is the core function for all merges/joins.
# df.merge() and df.join() are convenient wrappers:
# - df.join() defaults to left join on df1's index and df2's index.
# - df.merge() defaults to inner join on columns, but can join on indexes.
# Use merge() for flexibility; join() can save typing in simple index-based left joins.

In [6]:
# -------------------------------
# 5) pivot_table & crosstab
# -------------------------------
# pivot_table: average temperature per sensor per hour of day
sensor_df['hour'] = sensor_df['timestamp'].dt.hour
pv = pd.pivot_table(sensor_df, values='temperature', index='sensor_id', columns='hour', aggfunc='mean')
print('\n--- pivot_table mean temperature by sensor x hour ---')
print(pv.iloc[:, :6]) # show first 6 hours as sample

# crosstab: count of readings per sensor per type
sensor_meta_joined = sensor_df.merge(meta, on='sensor_id')
ct = pd.crosstab(sensor_meta_joined['sensor_id'], sensor_meta_joined['type'])
print('\n--- crosstab counts sensor x type ---')
print(ct)


--- pivot_table mean temperature by sensor x hour ---
hour               0          1          2          3          4          5
sensor_id                                                                  
S1         12.210048  10.664322  10.720927  12.989877  14.358660  14.856285
S2          9.690962  10.132154  10.812555  12.572447  13.260633  14.696669
S3         10.254658  10.215027  10.674089  13.038813  12.860999  15.427306
S4          9.326482  10.789089  11.722838  13.088461  13.111283  14.757508
S5         12.241501  12.651350  11.540636  13.750657  14.472117  16.459888
S6         13.411637  12.972633  14.169101  15.001611  16.085027  17.580157

--- crosstab counts sensor x type ---
type       hygro  thermo  thermo-hygro
sensor_id                             
S1             0       0           144
S2             0       0           144
S3             0     144             0
S4             0       0           144
S5           144       0             0
S6             0     144 

In [ ]:
# -------------------------------
# 6) Merging and joining examples (all possibilities)
# -------------------------------
# inner merge (only sensors present in both)
m_inner = sensor_df.merge(meta, on='sensor_id', how='inner')
# left merge (keeps all readings)
m_left = sensor_df.merge(meta, on='sensor_id', how='left')
# right and outer
m_right = sensor_df.merge(meta, on='sensor_id', how='right')
m_outer = sensor_df.merge(meta, on='sensor_id', how='outer')

print('\nmerge sizes: inner={}, left={}, right={}, outer={}'.format(
len(m_inner), len(m_left), len(m_right), len(m_outer)
))

# join on index: prepare two DataFrames with indexes
df_a = sensor_df.sample(50, random_state=4).set_index(['sensor_id', 'timestamp']).sort_index()
df_b = pd.DataFrame({'battery': np.random.randint(20, 100, size=30)},
index=df_a.index[:30])
# left join using index
joined = df_a.join(df_b, how='left')

print('\n--- joined (index join) sample ---')
print(joined.head())

# concat: stack vertically and horizontally
concat_v = pd.concat([sensor_df.head(5), sensor_df.tail(5)], axis=0)
concat_h = pd.concat([sensor_df.head(5).reset_index(drop=True).iloc[:, :3],
sensor_df.head(5).reset_index(drop=True).iloc[:, 3:]], axis=1)

# combine_first: combine two sources preferring left's non-null values
left = sensor_df.head(10).copy()
right = left.copy()
right.loc[right.sample(frac=0.3, random_state=5).index, 'temperature'] = np.nan
combined = right.combine_first(left)
print('\n--- combine_first example ---')
print(combined.head())

In [ ]:
# -------------------------------
# 7) Reshaping: stack/unstack and melt
# -------------------------------
# create a small pivot and show stack/unstack
small = sensor_df[sensor_df['sensor_id'].isin(['S1', 'S2'])].head(12)
pv_small = small.pivot(index='timestamp', columns='sensor_id', values='temperature')
print('\n--- pv_small pivot (timestamp x sensor) ---')
print(pv_small.head())
print('\nstacked -> unstacked roundtrip (demonstration)')
stacked = pv_small.stack()
print(stacked.head())
print(stacked.unstack().head())

# melt: convert wide back to long
melted = pv_small.reset_index().melt(id_vars='timestamp', var_name='sensor_id', value_name='temperature')
print('\n--- melted back to long ---')
print(melted.head())

In [ ]:
# -------------------------------
# 8) Query: expressive filtering
# -------------------------------
# Use DataFrame.query for readable boolean expressions
q = sensor_df.query('temperature > 25 and humidity < 50')
print(f'\nNumber of readings with temperature>25 and humidity<50: {len(q)}')

# boolean indexing vs query for more complex expressions
expr = "(temperature > temperature.mean()) & (humidity < humidity.quantile(0.25))"
# careful: can't use temperature.mean() inside query easily; compute first
temp_mean = sensor_df['temperature'].mean()
hum_q25 = sensor_df['humidity'].quantile(0.25)
q2 = sensor_df.query('@temp_mean < temperature and humidity < @hum_q25')
print(f'\nUsing external variables in query, matches: {len(q2)}')

In [ ]:
# -------------------------------
# 9) Aggregations with groupby + transform and filter
# -------------------------------
# find sensors whose median temperature is > global median
global_median = sensor_df['temperature'].median()
sensor_medians = sensor_df.groupby('sensor_id')['temperature'].median()
hot_sensors = sensor_medians[sensor_medians > global_median].index.tolist()
print('\nSensors with median temperature > global median:', hot_sensors)

# use transform to broadcast group statistic back to rows
sensor_df['sensor_temp_median'] = sensor_df.groupby('sensor_id')['temperature'].transform('median')
# filter rows where temperature is above group's median
above_group_median = sensor_df[sensor_df['temperature'] > sensor_df['sensor_temp_median']]
print('\nRows above their sensor median (sample):')
print(above_group_median.head())

# groupby.filter to keep only sensors with at least 200 readings
sensors_with_enough = sensor_df.groupby('sensor_id').filter(lambda g: len(g) >= 200)
print('\nSensors kept (len>=200):', sensors_with_enough['sensor_id'].unique())

In [ ]:
# -------------------------------
# 10) MultiIndex columns: produce aggregated table with hierarchical columns and plot
# -------------------------------
agg_multi = sensor_df.groupby('sensor_id').agg(
temp_mean=('temperature', 'mean'),
temp_std=('temperature', 'std'),
hum_mean=('humidity', 'mean'),
hum_std=('humidity', 'std')
)
# create MultiIndex columns manually
agg_multi.columns = pd.MultiIndex.from_tuples([('temperature', 'mean'), ('temperature', 'std'), ('humidity', 'mean'), ('humidity', 'std')])
print('\n--- agg_multi (MultiIndex columns) ---')
print(agg_multi)

In [ ]:
# -------------------------------
# 11) Rolling, expanding and time-based resampling (for sensor S2)
# -------------------------------
s2 = sensor_df[sensor_df['sensor_id'] == 'S2'].set_index('timestamp').sort_index()
# rolling 30-minute window (freq is 10min so 3 observations -> use time-based window)
s2['temp_roll_30min'] = s2['temperature'].rolling('30min').mean()
# humidity diff next reading -> shift(-1)
s2['humidity_next_diff'] = s2['humidity'].shift(-1) - s2['humidity']
# boolean: temp increased >5C and humidity dropped >20 compared to previous measurement
s2['temp_prev_diff'] = s2['temperature'] - s2['temperature'].shift(1)
s2['hum_prev_diff'] = s2['humidity'].shift(1) - s2['humidity']
s2['extreme_change'] = (s2['temp_prev_diff'] > 5) & (s2['hum_prev_diff'] > 20)

    ## Task Specification: Aggregations, Joins, GroupBy, and Rolling Analysis with Pandas

---

### **Task Overview**
This assignment simulates a **customer purchase analytics workflow** using Pandas. You will work with timestamped transaction data and customer metadata, performing:

- Group-based aggregations
- Time-based analysis
- Rolling window calculations within groups
- Joining customer metadata from another dataset

---

### **Datasets**
You are provided with two datasets:

| File | Description |
|------|-------------|
| `data/purchases.csv` | Timestamped purchase data (contains `Customer_id`, `Timestamp`, `Price`, etc.) |
| `data/customer_info.csv` | Customer metadata (contains `Customer_id`, `Born_date`, `City`, etc.) |

---

### **1. Load and Prepare the Data**
- **Task 1.1:** Load the two CSV files into Pandas DataFrames:
  - Use `df` for `purchases.csv`
  - Use `customer_info` for `customer_info.csv`
  - Shuffle the rows of `df` right after loading: `df = df.sample(frac=1, random_state=42)`

- **Task 1.2:** Convert the `Timestamp` column in `df` to Pandas `datetime` format.

- **Task 1.3:** Set `Timestamp` as the **index** of `df` and **sort** the index in ascending order.

---

### **2. Join Customer Metadata**
- **Task 2.1:** Merge the purchase DataFrame (`df`) with the customer metadata DataFrame (`customer_info`) on the `Customer_id` column.

- **Task 2.2:** The resulting DataFrame should include the columns `Born_date` and `City` from the `customer_info` DataFrame.

---

### **3. GroupBy and Aggregations**
Using the merged DataFrame:

- **Task 3.1:** Group by **Customer_id** and compute:
  - Total revenue (`sum(Price)`)
  - Number of purchases (`count`)
  - Average purchase price (`mean(Price)`)

- **Task 3.2:** Create a new feature by extracting the **month name** (e.g., `"January"`, `"February"`) from the `Timestamp`.

- **Task 3.3:** Group by **City** and **Month** and compute:
  - Total monthly revenue per city (`sum(Price)`)

---

### **4. Rolling Window Analysis (Per Customer)**
Perform rolling window calculations **within each customer group**:

- Inside each customer group:
  - **Sort** the DataFrame by `Timestamp`.

- Compute the following:
  - A **3-purchase rolling average** of `Price`.
  - A `diff()` column showing the change in price compared to the previous purchase.

> **Note:** The rolling operations must be computed **within each customer group**, not across all customers.

---

### **5. Advanced Filtering**
- Do the same as in 4 only for customers from **Berlin**, make sure only necessary values are joined (that is filter first, join second).

- Do the same as in 4 only for rows where the `Price` of the current purchase is greater than **20%** than the previous purchase (`Price`), make sure only necessary values are joined (that is filter first, join second).

---

### **6. Custom Lambda Function for GroupBy Operations**
- Apply a **custom lambda function** within a `groupby()` operation to filter or modify the data. For example:
  - Use a lambda to **filter** customers who have purchased more than a specific amount within a given timeframe (e.g., greater than $1000 in total purchases during the last quarter).
  - Apply custom transformations to grouped data using the lambda function.

---

### 7. Time-Based Analysis
- Perform additional time-based analysis:
  - Calculate **monthly growth rate** for total revenue per customer (i.e., compare total revenue of the current month with the previous month): `growth_rate = (revenue_current_month - revenue_previous_month) / revenue_previous_month` (`NaN` for the first month).
  - Create a **cumulative sum** of purchases over time per customer.


In [91]:
import numpy as np
import pandas as pd
#load data, create data frames:
df = pd.read_csv('data/purchases.csv')
customer_info = pd.read_csv('data/customer_info.csv')

#shuffle
df = df.sample(frac=1, random_state=42)

#datetime format
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
print(df[:5])

#set time as index and sort
df = df.set_index(['Timestamp']).sort_index()
print(df[:5])

     Customer_id           Timestamp   Price
1801        C050 2025-02-07 12:30:00  142.66
1190        C030 2025-01-25 19:00:00   61.57
1817        C011 2025-02-07 20:30:00   29.86
251         C023 2025-01-06 05:30:00  366.28
2505        C021 2025-02-22 04:30:00  456.30
                    Customer_id   Price
Timestamp                              
2025-01-01 00:00:00        C037  436.71
2025-01-01 00:30:00        C049  274.70
2025-01-01 01:00:00        C017  380.42
2025-01-01 01:30:00        C049  378.28
2025-01-01 02:00:00        C002   13.88


In [92]:
#merge dataframes:
df_reset = df.reset_index() #musíme resetovat index, abychom neztratili datum!

merge_1 = df_reset.merge(customer_info, on='Customer_id', how='inner')
#print(merge_1)

#groupby
agg_funcs = {
'Price': ['sum','count','mean']
}

grouped = merge_1.groupby('Customer_id').agg(agg_funcs)
print(grouped[:5])

#month (pomocí funkce month_name)
merge_1['Month'] = merge_1['Timestamp'].dt.month_name()
print(merge_1)

#City
agg_funcs2 = {
'Price': ['sum']
}
grouped2 = merge_1.groupby(['City', 'Month']).agg(agg_funcs2)
print(grouped2)

                Price                  
                  sum count        mean
Customer_id                            
C001         13055.75    56  233.138393
C002         15927.80    64  248.871875
C003         14947.34    56  266.916786
C004         15111.43    59  256.125932
C005         16798.61    64  262.478281
               Timestamp Customer_id   Price   Born_date       City    Month
0    2025-01-01 00:00:00        C037  436.71  1996-09-14     Munich  January
1    2025-01-01 00:30:00        C049  274.70  1979-01-20     Munich  January
2    2025-01-01 01:00:00        C017  380.42  1981-02-23    Cologne  January
3    2025-01-01 01:30:00        C049  378.28  1979-01-20     Munich  January
4    2025-01-01 02:00:00        C002   13.88  1988-07-03  Frankfurt  January
...                  ...         ...     ...         ...        ...      ...
2995 2025-03-04 09:30:00        C010  147.50  1970-04-07    Hamburg    March
2996 2025-03-04 10:00:00        C035  363.72  2003-12-26  Frankf

In [98]:
#setrdidime
merge_1 = merge_1.sort_values('Timestamp')
merge_1
#prumer 3 po sobě
merge_1['Rolling_Price_Average'] = merge_1.groupby('Customer_id')['Price'].transform(lambda x: x.rolling(window=3).mean())

#rozdil
merge_1['Price_Diff'] = merge_1.groupby('Customer_id')['Price'].diff()
merge_1

,Timestamp,Customer_id,Price,Born_date,City,Month,Rolling_Price_Avg,Rolling_Price_Average,Price_Diff
0,2025-01-01 00:00:00,C037,436.71,1996-09-14,Munich,January,NaN,NaN,NaN
1,2025-01-01 00:30:00,C049,274.70,1979-01-20,Munich,January,NaN,NaN,NaN
2,2025-01-01 01:00:00,C017,380.42,1981-02-23,Cologne,January,NaN,NaN,NaN
3,2025-01-01 01:30:00,C049,378.28,1979-01-20,Munich,January,NaN,NaN,103.58
4,2025-01-01 02:00:00,C002,13.88,1988-07-03,Frankfurt,January,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
2995,2025-03-04 09:30:00,C010,147.50,1970-04-07,Hamburg,March,99.070000,99.070000,15.05
2996,2025-03-04 10:00:00,C035,363.72,2003-12-26,Frankfurt,March,342.810000,342.810000,95.70
2997,2025-03-04 10:30:00,C049,419.03,1979-01-20,Munich,March,376.723333,376.723333,-30.66
2998,2025-03-04 11:00:00,C032,431.93,1987-10-03,Munich,March,354.903333,354.903333,180.79


In [110]:
berlin_customers = customer_info[customer_info['City'] == 'Berlin']

#mergneme jen ty z berlína
df_reset = df.reset_index()
merge_berlin = df_reset.merge(berlin_customers, on='Customer_id', how='inner')

#setrdidime
merge_berlin = merge_1[merge_1['City'] == "Berlin"]

merge_berlin = merge_berlin.sort_values('Timestamp')
merge_berlin
#prumer 3 po sobě
merge_berlin['Rolling_Price_Average'] = merge_berlin.groupby('Customer_id')['Price'].transform(lambda x: x.rolling(window=3).mean())

#rozdil
merge_berlin['Price_Diff'] = merge_berlin.groupby('Customer_id')['Price'].diff()
merge_berlin


#price_increase_20:
#novy df
df_ready = df.sort_index().reset_index()

#predchozi cena
df_ready['Prev_Price'] = df_ready.groupby('Customer_id')['Price'].shift(1)

#kde je narust o 20%
df_filtered = df_ready[df_ready['Price'] > 1.20 * df_ready['Prev_Price']].copy()

#mergneme
merge_filtered = df_filtered.merge(customer_info, on='Customer_id', how='inner')

#prumery
merge_filtered['Rolling_Price_Avg'] = merge_filtered.groupby('Customer_id')['Price'].transform(lambda x: x.rolling(window=3).mean())

#rozdily
merge_filtered['Price_Diff'] = merge_filtered.groupby('Customer_id')['Price'].diff()

merge_filtered

,Timestamp,Customer_id,Price,Prev_Price,Born_date,City,Rolling_Price_Avg,Price_Diff
0,2025-01-01 01:30:00,C049,378.28,274.70,1979-01-20,Munich,NaN,NaN
1,2025-01-01 02:30:00,C002,67.50,13.88,1988-07-03,Frankfurt,NaN,NaN
2,2025-01-01 06:30:00,C002,278.49,67.50,1988-07-03,Frankfurt,NaN,210.99
3,2025-01-01 09:00:00,C032,400.18,144.80,1987-10-03,Munich,NaN,NaN
4,2025-01-01 14:00:00,C011,379.95,228.59,1970-02-09,Hamburg,NaN,NaN
...,...,...,...,...,...,...,...,...
1203,2025-03-04 06:30:00,C022,376.23,72.50,1997-04-12,Cologne,334.003333,18.90
1204,2025-03-04 08:30:00,C040,447.24,62.33,1968-07-09,Munich,317.593333,133.03
1205,2025-03-04 10:00:00,C035,363.72,268.02,2003-12-26,Frankfurt,298.186667,-32.97
1206,2025-03-04 11:00:00,C032,431.93,251.14,1987-10-03,Munich,406.436667,54.27


In [112]:
splnili_podminku = merge_1.groupby('Customer_id').filter(lambda x: x['Price'].sum() > 1000)

print(splnili_podminku)

               Timestamp Customer_id   Price   Born_date       City    Month  \
0    2025-01-01 00:00:00        C037  436.71  1996-09-14     Munich  January   
1    2025-01-01 00:30:00        C049  274.70  1979-01-20     Munich  January   
2    2025-01-01 01:00:00        C017  380.42  1981-02-23    Cologne  January   
3    2025-01-01 01:30:00        C049  378.28  1979-01-20     Munich  January   
4    2025-01-01 02:00:00        C002   13.88  1988-07-03  Frankfurt  January   
...                  ...         ...     ...         ...        ...      ...   
2995 2025-03-04 09:30:00        C010  147.50  1970-04-07    Hamburg    March   
2996 2025-03-04 10:00:00        C035  363.72  2003-12-26  Frankfurt    March   
2997 2025-03-04 10:30:00        C049  419.03  1979-01-20     Munich    March   
2998 2025-03-04 11:00:00        C032  431.93  1987-10-03     Munich    March   
2999 2025-03-04 11:30:00        C007  161.42  1998-04-07     Munich    March   

      Rolling_Price_Avg  Rolling_Price_

In [125]:

#mesicni sumy:
monthly_data = merge_1.groupby(['Customer_id', 'Month'])['Price'].sum().reset_index()
#kumulativni utraty
monthly_data['Cumulative_Spend'] = monthly_data.groupby('Customer_id')['Price'].cumsum()
#prirustky za mesic
monthly_data['Growth_rate'] = monthly_data.groupby('Customer_id')['Price'].diff()
monthly_data

,Customer_id,Month,Price,Cumulative_Spend,Growth_rate
0,C001,February,5204.79,5204.79,NaN
1,C001,January,7302.10,12506.89,2097.31
2,C001,March,548.86,13055.75,-6753.24
3,C002,February,7838.50,7838.50,NaN
4,C002,January,7538.37,15376.87,-300.13
...,...,...,...,...,...
144,C049,January,10952.12,16513.14,5391.10
145,C049,March,1607.60,18120.74,-9344.52
146,C050,February,6232.23,6232.23,NaN
147,C050,January,5379.40,11611.63,-852.83
